In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from config import get_config, latest_weights_file_path
from train_test import get_model, run_validation, get_ds
from translate import translate

c:\Users\nikol\Desktop\UNIBO\Terzo Anno\Tesi\Transformer\Transformer-\envTrans\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Max length of source sentence: 309
Max length of target sentence: 274


In [6]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

# Load the pretrained weights
model_filename = latest_weights_file_path(config)
state = torch.load(model_filename, map_location=device)
model.load_state_dict(state['model_state_dict'])

# ← aggiungi qui sotto
model.eval()
with torch.no_grad():
    for i, batch in enumerate(val_dataloader):
        if i == 10:
            break
        
        encoder_input = batch['encoder_input'].to(device)
        encoder_mask = batch['encoder_mask'].to(device)
        decoder_input = batch['decoder_input'].to(device)
        decoder_mask = batch['decoder_mask'].to(device)
        encoder_output = model.encode(encoder_input, encoder_mask)
        out = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
        proj = model.project(out)

        predicted_tokens = torch.argmax(proj, dim=-1)

        print(f"--- Frase {i+1} ---")
        print("SOURCE:   ", batch['src_text'][0])
        print("TARGET:   ", batch['tgt_text'][0])
        print("PREDICTED:", tokenizer_tgt.decode(predicted_tokens[0].cpu().numpy()))
        print()

Using device: cpu
Max length of source sentence: 309
Max length of target sentence: 274
--- Frase 1 ---
SOURCE:    That's to say, all Petersburg will be there...
TARGET:    Significa che tutta Pietroburgo è là.
PREDICTED: Significa che tutta Pietroburgo è là . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

--- Frase 2 ---
SOURCE:    He had really promised to go to Bryansky's, who lived seven miles from Peterhof, and pay him for the horses, and he hoped to make time to call there too.
TARGET:    Vronskij aveva davvero promesso di andare da Brjanskij a dieci verste da Petergof, a portargli il denaro per i cavalli; voleva trovare il tempo di andare anche là.
PREDICTED: Vronskij aveva davvero promesso di andare da Brjanskij a dieci verste da Petergof , a portargli il denaro per i cavalli ; voleva trovare il tempo di andare anche là . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

In [8]:
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: print(msg), 0, None, num_examples=10)

token generati: tensor([2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4, 7,
        4, 7, 4, 7, 4, 7

KeyboardInterrupt: 

In [4]:
t = translate("The old man walked slowly through the forest, his eyes fixed upon the distant mountains.")

Using device: cpu
encoder_mask shape: torch.Size([1, 1, 1, 350])
source_mask shape in translate: torch.Size([1, 1, 1, 350])
    SOURCE: The old man walked slowly through the forest, his eyes fixed upon the distant mountains.
 PREDICTED: , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , di di di di di di di di di 

KeyboardInterrupt: 

In [ ]:
# t = translate(34)

Using device: cpu


KeyError: 'datasource'